# 06a - Ablations: TLD memorisation and the tier C protocol

**Runtime -> Run all (GPU). Resumes** via saved predictions.

Two things notebook 06 revealed:

**A. Tier A generalisation.** XGBoost scored 0.9866 on the random split but
0.8992 family-disjoint - *below* logistic regression. The corpus audit flagged
`tld` as a likely family fingerprint (many generators emit into a fixed TLD).
Part A re-runs tier A **without tld** to test that directly.

**B. Tier C protocol.** Early stopping on a validation part with a handful of
positives stopped runs at 0-3 iterations, and the family-disjoint split does
not hold out the live-malicious sources. Tier C gets its own protocol:

* **Repeated stratified 5-fold CV (3 repeats)** over the full cert-bearing
  population, regularised fixed-round models, no early stopping. Out-of-fold
  predictions are pooled per repeat; mean +/- std over repeats.
* **Source-disjoint evaluation**: train on URLhaus positives, test on
  OpenPhish (and the reverse). Generalisation across *feeds* is the right
  test for live infrastructure, where DGA-family disjointness is irrelevant.
* **Lexical-only control inside tier C.** The ablation that isolates what
  certificate *properties* add - the paper's novel claim rests on this row.

In [ ]:
# --- standard header ---
from google.colab import drive
drive.mount('/content/drive')

import os, sys, subprocess, getpass
REPO = '/content/secure-dns-trust-ai'
URL  = 'github.com/sandesh20lamichhane/secure-dns-trust-ai.git'
if os.path.isdir(REPO):
    subprocess.run(['git','-C',REPO,'pull','-q'], check=False)
else:
    TOKEN = getpass.getpass('GitHub PAT: ')
    subprocess.run(['git','clone','-q',f'https://{TOKEN}@{URL}',REPO], check=True)
sys.path.insert(0, REPO)
os.environ['DNSTRUST_CONFIG_DIR'] = f'{REPO}/configs'

from src.utils import config, manifest, seeds
P = config.paths(); config.ensure_tree(P); seeds.set_all(42)
print('repo', manifest.git_sha(REPO))

In [ ]:
!pip -q install pyarrow zstandard xgboost

In [ ]:
import pandas as pd, numpy as np, xgboost as xgb
from pathlib import Path
from src.evaluate import splits, metrics, predictions
from src.utils import manifest as mf

DEVICE = 'cuda'
try:
    xgb.XGBClassifier(device='cuda', n_estimators=1).fit(np.zeros((4,2),np.float32),[0,1,0,1])
except Exception:
    DEVICE = 'cpu'
print('device:', DEVICE)

X_all  = pd.read_parquet(f"{P['data']['features']}/fused_v1.parquet")
labels = pd.read_parquet(f"{P['data']['interim']}/domains_labelled.parquet")[['domain','source','family']]
X_all  = X_all.merge(labels, on='domain', how='left')
PRED_DIR, MODEL_DIR = Path(P['artifacts']['predictions']), Path(P['artifacts']['models'])

LEXICAL = ['length','core_length','n_labels','shannon_entropy','vowel_ratio',
           'digit_ratio','hyphen_count','max_consec_consonants','bigram_score',
           'trigram_score','unique_char_ratio','is_idn','has_digit','starts_with_digit']
CERT_PROPS = ['is_free_ca','validity_days','days_until_expiry','cert_age_days',
              'is_expired','is_not_yet_valid','is_self_signed','san_count',
              'wildcard_san','cn_in_san','key_bits','short_validity',
              'very_fresh_cert','cn_san_mismatch','weak_key']
CERT_CATS = ['issuer_org','key_algorithm','sig_algorithm','san_bucket']

def train_categories(df, cats): return {c: pd.Index(df[c].dropna().unique()) for c in cats}
def matrix(df, num, cats, categories):
    Xm = df[num].apply(pd.to_numeric, errors='coerce').astype(np.float32)
    for c in cats:
        known = df[c].where(df[c].isin(categories[c]))
        Xm[c] = pd.Categorical(known, categories=categories[c])
    return Xm, df['label'].values, df['domain'].values

## Part A - tier A without tld

In [ ]:
SEEDS = [42, 43, 44]
for split_name in ['random_v1', 'family_disjoint_v1']:
    sp = splits.load_split(P['data']['splits'], split_name); d = sp['domains']
    tr = X_all[X_all['domain'].isin(d['train'])]
    va = X_all[X_all['domain'].isin(d['val'])]
    te = X_all[X_all['domain'].isin(d['test'])]
    Xtr, ytr, _ = matrix(tr, LEXICAL, [], {}); Xva, yva, _ = matrix(va, LEXICAL, [], {})
    Xte, yte, dte = matrix(te, LEXICAL, [], {})
    spw = float((ytr==0).sum()) / max((ytr==1).sum(), 1)
    for seed in SEEDS:
        run_id = f'xgb_tierA_notld_{split_name}_s{seed}'
        if (PRED_DIR/f'{run_id}.parquet').exists():
            print('SKIP (done)', run_id); continue
        m_ = xgb.XGBClassifier(objective='binary:logistic', eval_metric='aucpr',
                tree_method='hist', device=DEVICE, max_depth=8, learning_rate=0.05,
                n_estimators=2000, early_stopping_rounds=100, subsample=0.8,
                colsample_bytree=0.8, min_child_weight=5, scale_pos_weight=spw,
                random_state=seed, verbosity=0)
        m_.fit(Xtr, ytr, eval_set=[(Xva, yva)], verbose=False)
        sc = m_.predict_proba(Xte)[:,1]; m = metrics.evaluate(yte, sc)
        m['best_iteration'] = int(m_.best_iteration)
        predictions.save(run_id, PRED_DIR, dte, yte, sc)
        m_.save_model(str(MODEL_DIR/f'{run_id}.json'))
        mf.record(P['manifest'], run_id, 'xgb_tierA_notld',
                  {'features': LEXICAL, 'ablation': 'tld removed'},
                  split_name, sp['split_file'], m, seed, repo_root=REPO)
        print(f'DONE {run_id:44s} roc={m["roc_auc"]:.4f} fpr@95={m["fpr_at_95_tpr"]:.4f} iters={m["best_iteration"]}')

In [ ]:
man = mf.load_manifest(P['manifest'])
a = man[man['run_family'].isin(['xgb_tierA_lexical','xgb_tierA_notld'])]
piv = (a.groupby(['run_family','split_name'])['metrics.roc_auc']
        .agg(['mean','std']).round(4).unstack('split_name'))
display(piv)
with_tld = a[a.run_family=='xgb_tierA_lexical'].groupby('split_name')['metrics.roc_auc'].mean()
no_tld   = a[a.run_family=='xgb_tierA_notld'].groupby('split_name')['metrics.roc_auc'].mean()
print('generalisation drop WITH tld   :', round(with_tld['random_v1']-with_tld['family_disjoint_v1'],4))
print('generalisation drop WITHOUT tld:', round(no_tld['random_v1']-no_tld['family_disjoint_v1'],4))

## Part B - tier C protocol

Population: every domain that serves a certificate. Positives are live
malicious infrastructure from URLhaus and OpenPhish plus the few DGA domains
that happen to resolve. Three feature sets are compared under an identical
protocol, so the differences are attributable:

| set | features | question |
|---|---|---|
| lexical | lexical only | what does the name alone give, within cert-holders? |
| cert | certificate properties only | what do certificate properties alone give? |
| lexcert | both | the fusion row |

In [ ]:
from sklearn.model_selection import RepeatedStratifiedKFold

C = X_all[X_all['has_certificate'] == True].copy().reset_index(drop=True)
print('tier C population:', len(C), '| positives:', int(C.label.sum()),
      f'| prevalence {C.label.mean():.4f}')
print(C[C.label==1]['source'].value_counts().to_dict())

FEATSETS = {
    'lexical': (LEXICAL, []),
    'cert':    (CERT_PROPS, CERT_CATS),
    'lexcert': (LEXICAL + CERT_PROPS, CERT_CATS),
}
# Regularised, fixed rounds: no early stopping on a positive-starved val part.
XGB_C = dict(objective='binary:logistic', tree_method='hist', device=DEVICE,
             enable_categorical=True, max_depth=4, learning_rate=0.05,
             n_estimators=400, subsample=0.8, colsample_bytree=0.8,
             min_child_weight=10, reg_lambda=5.0, verbosity=0)

In [ ]:
N_REPEATS, N_FOLDS = 3, 5
rskf = RepeatedStratifiedKFold(n_splits=N_FOLDS, n_repeats=N_REPEATS, random_state=42)
folds = list(rskf.split(C, C['label']))

for fs_name, (num, cats) in FEATSETS.items():
    for rep in range(N_REPEATS):
        run_id = f'xgb_tierC_cv_{fs_name}_rep{rep}'
        if (PRED_DIR/f'{run_id}.parquet').exists():
            print('SKIP (done)', run_id); continue
        oof = np.zeros(len(C), dtype=np.float32)
        for k in range(N_FOLDS):
            tr_idx, te_idx = folds[rep*N_FOLDS + k]
            tr, te = C.iloc[tr_idx], C.iloc[te_idx]
            categories = train_categories(tr, cats)
            Xtr, ytr, _ = matrix(tr, num, cats, categories)
            Xte, _, _   = matrix(te, num, cats, categories)
            spw = float((ytr==0).sum()) / max((ytr==1).sum(), 1)
            m_ = xgb.XGBClassifier(scale_pos_weight=spw, random_state=42+rep, **XGB_C)
            m_.fit(Xtr, ytr, verbose=False)
            oof[te_idx] = m_.predict_proba(Xte)[:,1]
        m = metrics.evaluate(C['label'].values, oof)
        predictions.save(run_id, PRED_DIR, C['domain'].values, C['label'].values, oof,
                         extra={'source': C['source'].values})
        mf.record(P['manifest'], run_id, f'xgb_tierC_cv_{fs_name}',
                  {'features': num + cats, 'protocol': f'repeated stratified {N_FOLDS}-fold',
                   'xgb': XGB_C}, 'tierC_cv', None, m, 42+rep, repo_root=REPO)
        print(f'DONE {run_id:36s} pr={m["pr_auc"]:.4f} roc={m["roc_auc"]:.4f} '
              f'fpr@95={m["fpr_at_95_tpr"]:.4f} fpr@99={m["fpr_at_99_tpr"]:.4f}')

In [ ]:
man = mf.load_manifest(P['manifest'])
cv = man[man['run_family'].str.startswith('xgb_tierC_cv_')].copy()
cv['featureset'] = cv['run_family'].str.replace('xgb_tierC_cv_','')
tab = (cv.groupby('featureset')
         [['metrics.pr_auc','metrics.roc_auc','metrics.fpr_at_95_tpr','metrics.fpr_at_99_tpr','metrics.mcc']]
         .agg(['mean','std']).round(4))
display(tab)
print(f'prevalence floor for PR-AUC: {C.label.mean():.4f}')
tab.to_csv(Path(P['results']['tables'])/'table_tierC_cv.csv')

### Source-disjoint

Train on one feed's positives, test on the other's. Benign domains are split
70/30 at random so both sides have negatives. DGA cert-holders stay in the
training side (they are neither feed).

In [ ]:
rng = np.random.default_rng(42)
ben = C[C.label==0]; mal = C[C.label==1]
ben_mask = rng.random(len(ben)) < 0.7
ben_tr, ben_te = ben[ben_mask], ben[~ben_mask]

pairs = [('urlhaus','openphish'), ('openphish','urlhaus')]
for train_src, test_src in pairs:
    tr = pd.concat([ben_tr, mal[mal.source != test_src]])       # all except held-out feed
    te = pd.concat([ben_te, mal[mal.source == test_src]])
    for fs_name, (num, cats) in FEATSETS.items():
        run_id = f'xgb_tierC_srcdisjoint_{fs_name}_train-{train_src}_test-{test_src}'
        if (PRED_DIR/f'{run_id}.parquet').exists():
            print('SKIP (done)', run_id); continue
        categories = train_categories(tr, cats)
        Xtr, ytr, _ = matrix(tr, num, cats, categories); Xte, yte, dte = matrix(te, num, cats, categories)
        spw = float((ytr==0).sum()) / max((ytr==1).sum(), 1)
        scores = np.mean([xgb.XGBClassifier(scale_pos_weight=spw, random_state=s_, **XGB_C)
                            .fit(Xtr, ytr, verbose=False).predict_proba(Xte)[:,1]
                          for s_ in (42,43,44)], axis=0)       # 3-seed average
        m = metrics.evaluate(yte, scores)
        predictions.save(run_id, PRED_DIR, dte, yte, scores)
        mf.record(P['manifest'], run_id, f'xgb_tierC_srcdisjoint_{fs_name}',
                  {'features': num+cats, 'train_source': train_src, 'test_source': test_src},
                  f'srcdisjoint_{train_src}_to_{test_src}', None, m, 42, repo_root=REPO)
        print(f'{fs_name:8s} {train_src:9s}->{test_src:9s} n_pos={int(yte.sum()):3d} '
              f'pr={m["pr_auc"]:.4f} roc={m["roc_auc"]:.4f} fpr@95={m["fpr_at_95_tpr"]:.4f}')

In [ ]:
man = mf.load_manifest(P['manifest'])
sd = man[man['run_family'].str.startswith('xgb_tierC_srcdisjoint_')].copy()
sd['featureset'] = sd['run_family'].str.replace('xgb_tierC_srcdisjoint_','')
tab = sd.pivot_table(index='featureset', columns='split_name',
                     values=['metrics.pr_auc','metrics.roc_auc']).round(4)
display(tab)
tab.to_csv(Path(P['results']['tables'])/'table_tierC_source_disjoint.csv')

---

**How to read Part B.** The row that matters is `lexcert` vs `lexical` inside
tier C. If certificate properties add to lexical within the cert-holding
population - and hold up source-disjoint - the novel claim stands. If they do
not, that is reported as such; the tier B result (certificate *presence* plus
properties, family-invariant) stands regardless.

365 positives is the binding constraint on tier C's precision. The daily
URLhaus/OpenPhish snapshots are what relax it before submission.

**Next:** `07_cnn_bilstm`.